# 02. KV Cache Engineering: PagedAttention, RadixAttention, H2O, SnapKV, and YOCO

- **Category:** 15_compression_peft_and_speculative_inference / 02_adaptive_kv_speculative_and_merging
- **Lineage & Citations:** Kwon et al. (SOSP 2023, *Efficient Memory Management for Large Language Model Serving with PagedAttention*); Zhang et al. (NeurIPS 2023, *H2O: Heavy-Hitter Oracle for Efficient Generative Inference of Large Language Models*); Li et al. (2024, *SnapKV: LLM Knows What You are Looking for Before Generation*); Sun et al. (2024, *You Only Cache Once: Decoder-Decoder Architectures for Language Models*).
- **Purpose:** Implement virtual memory block management (PagedAttention), dynamic heavy-hitter token eviction (H2O), prompt observation-window compression (SnapKV), and cross-decoder shared global KV caches (YOCO).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Adaptive Compute Early Exit Token Merging and Frugalgpt Cascades](01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.ipynb) $\to$ `🔗 Downstream Syntheses:` [Speculative Decoding Lossless Rejection Sampling and Drafting](03_speculative_decoding_lossless_rejection_sampling_and_drafting.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Never evict historical tokens with pure global top-$k$ attention scores without unconditionally pinning a local recency window ($W \ge 32$); newly generated tokens have low accumulated scores simply due to recency and will be prematurely pruned, causing repetitive gibberish.

## Mathematical Foundations & Governing Equations

### 1. PagedAttention Virtual Address Translation
For logical token index $t$:
$$\text{block} = \lfloor t / B_s \rfloor, \quad \text{offset} = t \pmod{B_s}, \quad K[b, t] = \text{Pool}[\text{Table}[b, \text{block}], \text{offset}]$$

### 2. Heavy-Hitter Oracle (H2O) Dynamic Eviction
Given accumulated attention $S_t = \sum_{\tau=1}^t A_{:, \tau}$, keep budget $K = W + H$:
$$\text{Keep}(i) = \mathbb{I}(i \ge N - W) \lor \mathbb{I}(i \in \text{top-}H(S_t[:N-W]))$$

### 3. SnapKV Observation Window Pooling
$$I_j = \frac{1}{L_{\text{obs}}} \sum_{i=N_{\text{prompt}}-L_{\text{obs}}}^{N_{\text{prompt}}} A_{i, j}, \quad \text{Retain} = \text{top-}k(I_j)$$

### 4. YOCO Global Cross-Decoder Caching
$$h^{(l)} = h^{(l-1)} + \text{softmax}\left(\frac{h^{(l-1)} W_Q^{(l)} K_{\text{global}}^\top}{\sqrt{d_k}}\right) V_{\text{global}} W_O^{(l)}$$
Reduces KV cache footprint from $L \times 2 \times N \times d$ to $1 \times 2 \times N \times d$ ($L$-fold parameter reduction in cache).

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/02_kv_cache_paged_radix_h2o_snapkv_and_yoco.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x, w, b):
    return x @ w + b

def paged_kv_gather_and_attend(query, key_blocks, value_blocks, block_table, context_len, block_size=4):
    batch, _, dim = query.shape
    scale = 1.0 / math.sqrt(dim)
    attn_outs, attn_weights_list = [], []
    for b in range(batch):
        k_list, v_list = [], []
        for t in range(context_len):
            block_idx = int(block_table[b, t // block_size].item())
            offset = t % block_size
            k_list.append(key_blocks[block_idx, offset])
            v_list.append(value_blocks[block_idx, offset])
        k_seq, v_seq = torch.stack(k_list), torch.stack(v_list)
        scores = (query[b] @ k_seq.transpose(-1, -2)) * scale
        probs = F.softmax(scores, dim=-1)
        attn_outs.append(probs @ v_seq)
        attn_weights_list.append(probs)
    return torch.stack(attn_outs), torch.stack(attn_weights_list)

def h2o_evict_mask(cumulative_scores, local_window=4, heavy_budget=4):
    batch, seq_len = cumulative_scores.shape
    keep_mask = torch.zeros_like(cumulative_scores)
    recent_start = max(0, seq_len - local_window)
    keep_mask[:, recent_start:] = 1.0
    cand_len = recent_start
    if cand_len > 0 and heavy_budget > 0:
        cand_scores = cumulative_scores[:, :cand_len]
        k = min(heavy_budget, cand_len)
        _, top_indices = torch.topk(cand_scores, k=k, dim=-1, largest=True)
        for b in range(batch):
            keep_mask[b, top_indices[b]] = 1.0
    return keep_mask

def snapkv_select_mask(prompt_attention, obs_window=4, budget=6):
    batch, _, prompt_len = prompt_attention.shape
    importance = torch.mean(prompt_attention, dim=1)
    retain_mask = torch.zeros_like(importance)
    k = min(budget, prompt_len)
    _, top_idx = torch.topk(importance, k=k, dim=-1, largest=True)
    for b in range(batch):
        retain_mask[b, top_idx[b]] = 1.0
    return retain_mask

def yoco_cross_layer_forward(h_mid, k_shared, v_shared, w_q, w_out):
    _, _, dim = h_mid.shape
    scale = 1.0 / math.sqrt(dim)
    q = h_mid @ w_q
    scores = (q @ k_shared.transpose(-1, -2)) * scale
    attn = F.softmax(scores, dim=-1)
    return h_mid + (attn @ v_shared) @ w_out

In [ ]:
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-6

p_out, p_w = paged_kv_gather_and_attend(weights["query"], weights["key_blocks"], weights["value_blocks"], weights["block_table"], 10, 4)
assert float(torch.max(torch.abs(p_out - weights["paged_out"]))) < 1e-5
assert float(torch.max(torch.abs(p_w - weights["paged_weights"]))) < 1e-5

h_mask = h2o_evict_mask(weights["cumulative_scores"], 4, 3)
assert float(torch.max(torch.abs(h_mask - weights["h2o_mask"]))) < 1e-5

s_mask = snapkv_select_mask(weights["prompt_attn"], 4, 6)
assert float(torch.max(torch.abs(s_mask - weights["snapkv_mask"]))) < 1e-5

y_out = yoco_cross_layer_forward(weights["h_mid"], weights["k_shared"], weights["v_shared"], weights["w_q"], weights["w_out"])
assert float(torch.max(torch.abs(y_out - weights["yoco_out"]))) < 1e-5
print("All 5 KV cache components achieve perfect golden parity (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_paged = benchmark_fn(lambda: paged_kv_gather_and_attend(weights["query"], weights["key_blocks"], weights["value_blocks"], weights["block_table"], 10, 4), iters=100, name="PagedAttention Gather+Attend")
b_yoco = benchmark_fn(lambda: yoco_cross_layer_forward(weights["h_mid"], weights["k_shared"], weights["v_shared"], weights["w_q"], weights["w_out"]), iters=100, name="YOCO Global Cross-Attend")
print(format_benchmark_table([b_paged, b_yoco]))

retained = h_mask.sum().item()
total = h_mask.numel()
print(f"H2O Cache Footprint: {retained:.0f}/{total} tokens preserved ({(1.0 - retained/total)*100.0:.1f}% compression)")

In [ ]:
# Pathological Misuse: Pure Top-K Eviction without Local Window drops newly generated tokens!
scores = torch.tensor([[10.0, 9.0, 8.0, 7.0, 6.0, 0.1, 0.05, 0.01]])  # last 3 are brand-new tokens

# 1. Flawed pure top-k (budget 5): selects tokens 0, 1, 2, 3, 4 and EVICTS tokens 5, 6, 7!
_, top_flawed = torch.topk(scores, k=5, dim=-1)
flawed_mask = torch.zeros_like(scores)
flawed_mask.scatter_(1, top_flawed, 1.0)

# 2. Correct H2O: preserves local window of 3 (tokens 5, 6, 7) + top 2 heavy hitters (0, 1)
correct_mask = h2o_evict_mask(scores, local_window=3, heavy_budget=2)

assert flawed_mask[0, -1] == 0.0, "Flawed strategy incorrectly evicted the latest token!"
assert correct_mask[0, -1] == 1.0, "H2O correctly preserved the latest token via local recency window!"
print("Verified: Pure top-k destroys recent causal context; H2O recency window is strictly mandatory.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Eliminate Fragmentation via Pagination**: PagedAttention eliminates $>60\%$ memory waste by decoupling logical sequence length from physical contiguous HBM blocks.
2. **The Recency Guarantee**: Token eviction requires a hybrid policy (local recency window + heavy hitters); pure global top-$k$ causes immediate autoregressive breakdown.
3. **Architectural KV Reduction**: YOCO demonstrates that maintaining separate key-value caches across all $L$ layers is redundant; a single global intermediate KV suffices.

---
### Cross-Domain Generalisation Matrix
- **OS Virtual Memory:** PagedAttention directly mirrors page table translations, page faults, and block tables from virtual memory architecture.
- **Video Diffusion:** Frame generation caches spatial cross-attention states using observation window pooling (SnapKV).
- **Database Indexing:** RadixAttention directly maps common query prefix routing to trie-based in-memory database index lookups.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_kv_cache_paged_radix_h2o_snapkv_and_yoco.md](../../../markdown/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/02_kv_cache_paged_radix_h2o_snapkv_and_yoco.md)
- **PyTorch 2.x Module:** [02_kv_cache_paged_radix_h2o_snapkv_and_yoco.py](../../../pytorch2/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/02_kv_cache_paged_radix_h2o_snapkv_and_yoco.py)
- **Functional JAX Module:** [02_kv_cache_paged_radix_h2o_snapkv_and_yoco.py](../../../jax/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/02_kv_cache_paged_radix_h2o_snapkv_and_yoco.py)
- **Native Rust Candle Track:** [02_kv_cache_paged_radix_h2o_snapkv_and_yoco.rs](../../../candle/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/02_kv_cache_paged_radix_h2o_snapkv_and_yoco.rs)
- **Hardware Kernel (CPU / WGSL):** [02_kv_cache_paged_radix_h2o_snapkv_and_yoco.rs](../../../kernels/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/02_kv_cache_paged_radix_h2o_snapkv_and_yoco.rs)
- **Golden SafeTensors:** [02_kv_cache_paged_radix_h2o_snapkv_and_yoco.safetensors](../../../weights/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/02_kv_cache_paged_radix_h2o_snapkv_and_yoco.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. Adaptive Compute Early Exit Token Merging and Frugalgpt Cascades](01_adaptive_compute_early_exit_token_merging_and_frugalgpt_cascades.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 03. Speculative Decoding Lossless Rejection Sampling and Drafting](03_speculative_decoding_lossless_rejection_sampling_and_drafting.ipynb) &rarr;